# 02 · Pré-processamento do SINAN 2024 e definição do alvo

Guardiã AI · Tech Challenge 5 · grupo fiap-ssp-2025

Este notebook reproduz, com as funções do pacote `guardia`, a construção da tabela de treino a partir dos
microdados brutos do SINAN (violência interpessoal/autoprovocada, 2024, DATASUS). A extração DBC → parquet
vem do Tech Challenge 1 (`notebooks/01_extract_sinan_data.ipynb` daquele repositório).

**O que muda em relação às fases anteriores**
- O alvo deixa de ser `OUT_VEZES` ("ocorreu outras vezes?", campo retrospectivo da mesma ficha) e passa a ser
  `ENCAMINHA_PROTECAO`: a ficha registrou encaminhamento para delegacia da mulher, centro de atendimento à mulher,
  Ministério Público ou Defensoria. `OUT_VEZES` vira feature.
- Ausentes viram a categoria `IGNORADO` em vez de `dropna()`: treinamos com ~281 mil linhas, não 57 mil.
- Codificação nominal com `OneHotEncoder` dentro de um `Pipeline` (ver notebook 03), não `LabelEncoder`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from guardia.data.features import (
    FEATURE_NAMES, FEATURES, PROTECTION_COLS, TARGET, build_table, load_raw,
)

In [2]:
df_raw = load_raw()
print(f"bruto: {len(df_raw):,} notificações · {df_raw.shape[1]} colunas")

bruto: 662,803 notificações · 160 colunas


## Filtros e alvo
Mulheres (`CS_SEXO == F`) com lesão **não** autoprovocada (`LES_AUTOP == 2`). O alvo só existe quando ao menos
um dos quatro campos de encaminhamento foi respondido (Sim ou Não).

In [3]:
mulheres = df_raw[(df_raw["CS_SEXO"].str.strip() == "F") & (df_raw["LES_AUTOP"].str.strip() == "2")]
print(f"mulheres, não autoprovocada: {len(mulheres):,}")
for c in PROTECTION_COLS:
    vc = mulheres[c].fillna("").str.strip().replace("", "<vazio>").value_counts()
    print(f"{c:<11} {dict(vc.head(4))}")

mulheres, não autoprovocada: 294,393
DELEG_MULH  {'2': np.int64(230535), '1': np.int64(47902), '9': np.int64(11062), '<vazio>': np.int64(4894)}
ATEND_MULH  {'2': np.int64(239319), '1': np.int64(40208), '9': np.int64(10394), '<vazio>': np.int64(4472)}
MPU         {'2': np.int64(271942), '9': np.int64(10276), '1': np.int64(7046), '<vazio>': np.int64(5129)}
DEFEN_PUBL  {'2': np.int64(270566), '9': np.int64(10383), '1': np.int64(8312), '<vazio>': np.int64(5132)}


In [4]:
table = build_table(df_raw)
print(f"tabela: {len(table):,} linhas · {len(FEATURE_NAMES)} features · prevalência do alvo {table[TARGET].mean():.3f}")

tabela: 280,757 linhas · 42 features · prevalência do alvo 0.283


## Catálogo de features
Binárias (tipos de violência, meios de agressão, relação com o autor), nominais (`IGNORADO` explícito),
ordinal (escolaridade) e numérica (idade), cada uma com o rótulo usado no formulário e no relatório.

In [5]:
pd.DataFrame([{"feature": f.name, "tipo": f.kind, "grupo": f.group, "rótulo": f.label, "opções": ", ".join(f.options[:6])} for f in FEATURES])

,feature,tipo,grupo,rótulo,opções
0,VIOL_FISIC,binary,Tipo de violência,Violência física,
1,VIOL_PSICO,binary,Tipo de violência,Violência psicológica / moral,
2,VIOL_SEXU,binary,Tipo de violência,Violência sexual,
3,VIOL_TORT,binary,Tipo de violência,Tortura,
4,VIOL_FINAN,binary,Tipo de violência,Violência financeira / patrimonial,
5,VIOL_NEGLI,binary,Tipo de violência,Negligência / abandono,
6,VIOL_INFAN,binary,Tipo de violência,Trabalho infantil,
7,VIOL_LEGAL,binary,Tipo de violência,Intervenção legal,
8,VIOL_TRAF,binary,Tipo de violência,Tráfico de pessoas,
9,VIOL_OUTR,binary,Tipo de violência,Outro tipo de violência,


## Distribuições que importam

In [6]:
resumo = pd.DataFrame({
    "prevalência do alvo por OUT_VEZES": table.groupby("OUT_VEZES")[TARGET].mean().round(3),
    "n": table.groupby("OUT_VEZES").size(),
})
resumo

,prevalência do alvo por OUT_VEZES,n
OUT_VEZES,,
IGNORADO,0.170,47026
NAO,0.209,109225
SIM,0.390,124506


In [7]:
table.groupby("REL_PARCEIRO_INTIMO")[TARGET].agg(["mean", "size"]).round(3)

,mean,size
REL_PARCEIRO_INTIMO,,
0,0.189,173427
1,0.434,107330


In [8]:
table.groupby("CS_RACA")[TARGET].agg(["mean", "size"]).round(3).sort_values("size", ascending=False)

,mean,size
CS_RACA,,
PARDA,0.279,131711
BRANCA,0.278,106684
PRETA,0.342,28203
IGNORADO,0.222,8252
INDIGENA,0.180,3168
AMARELA,0.318,2739


In [9]:
print("idade ausente:", f"{table['IDADE_IGN'].mean():.1%}", "· escolaridade ausente:", f"{table['ESCOLARIDADE_IGN'].mean():.1%}")

idade ausente: 0.0% · escolaridade ausente: 35.9%


## Salvar

In [10]:
out = ROOT / "data" / "processed" / "sinan_encaminhamento.parquet"
out.parent.mkdir(parents=True, exist_ok=True)
table.to_parquet(out, index=False)
print("salvo em", out.relative_to(ROOT))

salvo em data/processed/sinan_encaminhamento.parquet
